# Semantic Init A/B Testing

Measures the impact of PPMI-SVD semantic embedding init on val_bpb.

**Setup:** 1xA100 on Colab, 5000 steps per run, comparing alpha values and init variants.

**What we're testing:**
1. Alpha sweep: 0 (disabled), 0.005 (current), 0.01, 0.05, 0.1, 0.5
2. Bolder variants: larger co-occurrence window, full SVD replacement (alpha=1.0 with no random component)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Setup

In [ ]:
!git clone https://github.com/openai/parameter-golf.git /content/parameter-golf 2>/dev/null || echo "Already cloned"
%cd /content/parameter-golf
!pip install -q sentencepiece huggingface-hub numpy zstandard
!python data/cached_challenge_fineweb.py --train-shards 10 --variant sp1024

## 2. Upload your train_gpt.py

Upload your local `train_gpt.py` (with zstd, EMA, warmdown changes) to replace the upstream version.

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()  # upload your train_gpt.py
for name, data in uploaded.items():
    dest = f"/content/parameter-golf/{name}"
    with open(dest, "wb") as f:
        f.write(data)
    print(f"Wrote {dest} ({len(data)} bytes)")

## 3. Run config

Adjust these to control how many steps per run and which alphas to test.

In [ ]:
STEPS = 5000
VAL_EVERY = 500

# Alpha sweep: 0 = disabled (control), rest are experimental values
ALPHA_VALUES = [0, 0.005, 0.01, 0.05, 0.1, 0.5]

# Common env vars for all runs (1xGPU, no wallclock cap so we hit STEPS exactly)
BASE_ENV = (
    f"ITERATIONS={STEPS} "
    f"VAL_LOSS_EVERY={VAL_EVERY} "
    "TRAIN_LOG_EVERY=100 "
    "MAX_WALLCLOCK_SECONDS=0 "
    "WARMUP_STEPS=20 "
    "WARMDOWN_ITERS=3500 "
    "EVAL_STRIDE=256 "
    "NUM_LAYERS=11 "
    "MLP_MULT=3 "
    "TRAIN_SEQ_LEN=2048 "
    "MUON_WD=0.04 "
    "QAT_BITS=6 "
    "QAT_START_FRAC=0.15 "
    "VOCAB_SIZE=1024 "
    "DATA_PATH=./data/datasets/fineweb10B_sp1024/ "
    "TOKENIZER_PATH=./data/tokenizers/fineweb_1024_bpe.model "
)

print(f"Will run {len(ALPHA_VALUES)} experiments, {STEPS} steps each, validating every {VAL_EVERY} steps")

## 4. Alpha sweep

Runs each alpha value sequentially. Each run writes its log to `logs/`.

In [ ]:
import subprocess, os

os.makedirs("logs", exist_ok=True)

for alpha in ALPHA_VALUES:
    run_id = f"alpha_{alpha}"
    cmd = (
        f"{BASE_ENV} "
        f"SEM_EMBED_ALPHA={alpha} "
        f"RUN_ID={run_id} "
        f"torchrun --standalone --nproc_per_node=1 train_gpt.py"
    )
    print(f"\n{'='*60}")
    print(f"Running: alpha={alpha}  (run_id={run_id})")
    print(f"{'='*60}")
    result = subprocess.run(cmd, shell=True, capture_output=False)
    if result.returncode != 0:
        print(f"WARNING: run alpha={alpha} exited with code {result.returncode}")

print("\nAll alpha sweep runs complete.")

## 5. Parse logs and compare

In [ ]:
import re
from collections import defaultdict

def parse_log(path):
    """Extract (step, val_bpb) pairs and final roundtrip bpb from a log file."""
    val_curve = []
    roundtrip_bpb = None
    with open(path) as f:
        for line in f:
            # Mid-training validation: "step:N/M val_loss:X val_bpb:Y"
            m = re.search(r"step:(\d+)/\d+ val_loss:[\d.]+ val_bpb:([\d.]+)", line)
            if m:
                val_curve.append((int(m.group(1)), float(m.group(2))))
            # Final roundtrip: "final_int6_zstd_roundtrip_exact val_loss:X val_bpb:Y"
            m = re.search(r"final_int6_zstd_roundtrip_exact val_loss:[\d.]+ val_bpb:([\d.]+)", line)
            if m:
                roundtrip_bpb = float(m.group(1))
    return val_curve, roundtrip_bpb

results = {}
for alpha in ALPHA_VALUES:
    log_path = f"logs/alpha_{alpha}.txt"
    if os.path.exists(log_path):
        curve, rt_bpb = parse_log(log_path)
        results[alpha] = {"curve": curve, "roundtrip_bpb": rt_bpb}
        final_raw = curve[-1][1] if curve else None
        print(f"alpha={alpha:<6}  final_raw_bpb={final_raw}  roundtrip_bpb={rt_bpb}")
    else:
        print(f"alpha={alpha:<6}  LOG NOT FOUND")

if not results:
    print("\nNo logs found. Run the alpha sweep first.")

## 6. Plot val_bpb curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: val_bpb curves over training
for alpha, data in sorted(results.items()):
    curve = data["curve"]
    if curve:
        steps, bpbs = zip(*curve)
        label = f"alpha={alpha}" if alpha > 0 else "alpha=0 (disabled)"
        ax1.plot(steps, bpbs, marker="o", markersize=3, label=label)

ax1.set_xlabel("Step")
ax1.set_ylabel("val_bpb")
ax1.set_title("Val BPB during training (lower is better)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: final roundtrip bpb bar chart
alphas_with_rt = [(a, d["roundtrip_bpb"]) for a, d in sorted(results.items()) if d["roundtrip_bpb"] is not None]
if alphas_with_rt:
    labels = [f"{a}" for a, _ in alphas_with_rt]
    bpbs = [b for _, b in alphas_with_rt]
    colors = ["#d62728" if a == 0 else "#1f77b4" for a, _ in alphas_with_rt]
    bars = ax2.bar(labels, bpbs, color=colors)
    ax2.set_xlabel("sem_embed_alpha")
    ax2.set_ylabel("Roundtrip val_bpb")
    ax2.set_title("Post-quantization BPB (competition metric)")
    ax2.grid(True, alpha=0.3, axis="y")

    # Annotate bars
    for bar, bpb in zip(bars, bpbs):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f"{bpb:.4f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("semantic_init_ab_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: semantic_init_ab_results.png")

## 7. Early convergence analysis

The most likely benefit of semantic init is faster early convergence. This checks whether higher alpha values reach a given bpb threshold in fewer steps.

In [ ]:
THRESHOLDS = [1.40, 1.35, 1.30, 1.28, 1.26]

print(f"{'Alpha':<10}", end="")
for t in THRESHOLDS:
    print(f"{'< '+str(t):<12}", end="")
print()
print("-" * (10 + 12 * len(THRESHOLDS)))

for alpha, data in sorted(results.items()):
    curve = data["curve"]
    label = f"{alpha}" if alpha > 0 else "0 (off)"
    print(f"{label:<10}", end="")
    for t in THRESHOLDS:
        hit_step = None
        for step, bpb in curve:
            if bpb < t:
                hit_step = step
                break
        print(f"{hit_step if hit_step else 'never':<12}", end="")
    print()

## 8. (Optional) Bolder variant: co-occurrence window sweep

The default PPMI window is 5 tokens. A wider window captures longer-range co-occurrence but blurs local structure. Test window=5, 10, 20 at the best alpha from above.

Edit `BEST_ALPHA` below based on your results from the sweep above.

In [ ]:
BEST_ALPHA = 0.05  # <-- edit this based on your alpha sweep results
WINDOW_VALUES = [5, 10, 20]

for window in WINDOW_VALUES:
    run_id = f"window_{window}"
    cmd = (
        f"{BASE_ENV} "
        f"SEM_EMBED_ALPHA={BEST_ALPHA} "
        f"SEM_EMBED_WINDOW={window} "
        f"RUN_ID={run_id} "
        f"torchrun --standalone --nproc_per_node=1 train_gpt.py"
    )
    print(f"\n{'='*60}")
    print(f"Running: window={window}, alpha={BEST_ALPHA}  (run_id={run_id})")
    print(f"{'='*60}")
    result = subprocess.run(cmd, shell=True, capture_output=False)
    if result.returncode != 0:
        print(f"WARNING: run window={window} exited with code {result.returncode}")

print("\nWindow sweep complete.")

In [ ]:
# Parse and plot window sweep results
fig, ax = plt.subplots(figsize=(8, 5))

for window in WINDOW_VALUES:
    log_path = f"logs/window_{window}.txt"
    if os.path.exists(log_path):
        curve, rt_bpb = parse_log(log_path)
        if curve:
            steps, bpbs = zip(*curve)
            ax.plot(steps, bpbs, marker="o", markersize=3, label=f"window={window}")
        print(f"window={window:<4}  final_raw_bpb={curve[-1][1] if curve else None}  roundtrip_bpb={rt_bpb}")

# Also plot the alpha=0 baseline for reference
base_log = "logs/alpha_0.txt"
if os.path.exists(base_log):
    curve, _ = parse_log(base_log)
    if curve:
        steps, bpbs = zip(*curve)
        ax.plot(steps, bpbs, marker="x", markersize=3, linestyle="--", color="gray", label="no semantic init")

ax.set_xlabel("Step")
ax.set_ylabel("val_bpb")
ax.set_title(f"Co-occurrence window size (alpha={BEST_ALPHA})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("window_sweep_results.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Summary

Key questions this notebook answers:
1. **Does semantic init help at all?** Compare alpha=0 vs best alpha — look at both the curves and final roundtrip bpb.
2. **Does it help early or late?** If curves converge by step 5000, the optimizer washes it out. If the gap persists, it's a real contribution.
3. **Is there an optimal alpha?** Too small = no effect, too large = could hurt by biasing init too strongly.
4. **Does window size matter?** Wider windows capture more structure but may blur local patterns.